# LangChain 1.0 Fundamentals

This notebook covers the 6 essential building blocks of LangChain 1.0+:

1. **Agents** — `create_agent()` API
2. **Models** — `init_chat_model()` universal factory
3. **Messages** — The 4 message types
4. **Tools** — `@tool` decorator and dependency injection
5. **Short-term Memory** — Checkpointers and thread isolation
6. **Streaming** — Real-time output modes

**Docs**: [docs.langchain.com](https://docs.langchain.com/)

## Setup

In [1]:
%pip install -qU langchain langchain-openai langchain-tavily langgraph python-dotenv

Error processing line 1 of /Users/greatmaster/miniconda3/envs/oreilly-langchain/lib/python3.12/site-packages/distutils-precedence.pth:

  Traceback (most recent call last):
    File "<frozen site>", line 206, in addpackage
    File "<string>", line 1, in <module>
  ModuleNotFoundError: No module named '_distutils_hack'

Remainder of file ignored
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

# Verify required keys are set
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your .env file"
print("Environment ready!")

Environment ready!


---
## 1. Agents — `create_agent()`

The central API in LangChain 1.0. `create_agent()` replaces the old `AgentExecutor` pattern.

It returns a **LangGraph `CompiledGraph`** that you invoke with `{"messages": [...]}`.

```python
from langchain.agents import create_agent

agent = create_agent(
    model,              # str like "openai:gpt-4o-mini" or a ChatModel instance
    tools=None,         # list of @tool-decorated functions
    system_prompt=None,        # system prompt string
    checkpointer=None,  # for memory persistence
    response_format=None,  # for structured output
)
```

In [3]:
from langchain.agents import create_agent

# Minimal agent — 3 lines
agent = create_agent(model="openai:gpt-4o-mini", tools=[])
result = agent.invoke({"messages": "What is LangChain?"})
result["messages"][-1].pretty_print()

================================== Ai Message ==================================

LangChain is a framework designed to simplify the development of applications that utilize large language models (LLMs). It provides tools and abstractions that allow developers to create, manage, and deploy applications that involve natural language processing and generation.

Key features of LangChain include:

1. **Prompt Management**: It provides ways to create, manage, and optimize prompts to get better responses from language models.

2. **Chains**: LangChain allows for the creation of sequences of calls to LLMs or other components, enabling more complex workflows by chaining multiple actions together.

3. **Memory**: Support for managing conversational memory, enabling applications to maintain context across multiple interactions.

4. **Integrations**: LangChain can integrate with various data sources, APIs, and services, enabling developers to build applications that can pull in external data or p

### Agent with Tools

Agents become powerful when given tools. The agent decides **when** and **how** to use them.

In [8]:
from langchain_tavily import TavilySearch

search = TavilySearch(max_results=3)

agent = create_agent(
    model="openai:gpt-4o-mini",
    tools=[search],
    system_prompt="You are a helpful research assistant. Search the web when needed.",
)

result = agent.invoke({"messages": "What is the latest news about LangChain 1.0?"})
result["messages"][-1].pretty_print()

================================== Ai Message ==================================

It seems that there hasn't been any specific news about LangChain 1.0 in the past month. However, here are some recent articles that may be of interest:

1. **OpenAI launches GPT-5.3 Instant**: This update focuses on improving the general-purpose model that supports ChatGPT, enhancing response quality and conversational flow. [Read more here.](https://thenextweb.com/news/openai-launches-gpt-5-3-instant-to-improve-chatgpts-most-used-model)

2. **NC State basketball coaching hot board**: This article discusses potential coaching candidates for NC State basketball, although it's not related to LangChain. [Read here.](https://247sports.com/college/north-carolina-state/longformarticle/nc-state-basketball-coaching-hot-board-10-early-candidates-to-replace-will-wade-279135560/)

3. **Cursor's new programming model**: While not related to LangChain, it discusses a new model outperforming others in performance and 

In [9]:
# Stream the agent's steps to see its reasoning
for step in agent.stream(
    {"messages": "Who won the last FIFA World Cup?"},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    print("---")

================================ Human Message =================================

Who won the last FIFA World Cup?
---
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_72At1lpTbCh4GLsLy7Qw15im)
 Call ID: call_72At1lpTbCh4GLsLy7Qw15im
  Args:
    query: 2022 FIFA World Cup winner
---
================================= Tool Message =================================
Name: tavily_search

{"query": "2022 FIFA World Cup winner", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://sports.yahoo.com/articles/2022-world-cup-full-list-040136314.html", "title": "2022 World Cup: Full list of past winners year-by-year - Yahoo Sports", "content": "Argentina won the 2022 World Cup, further cementing Lionel Messi's name as one of, if not *the*, greatest players to ever play the game. Messi joins celebrated countryman Diego Maradona as debated 'GOATS' to lift the World Cup trophy, as Argentina won for th

---
## 2. Models — `init_chat_model()`

A universal factory for any LLM provider. Use the **string format** `"provider:model-name"` or initialize explicitly.

| Provider | String Format | Package |
|----------|--------------|----------|
| OpenAI | `"openai:gpt-5.4-mini"` | `langchain-openai` |
| Anthropic | `"anthropic:claude-sonnet-4-6"` | `langchain-anthropic` |
| Google | `"google_genai:gemini-3.1-flash"` | `langchain-google-genai` |

In [10]:
from langchain.chat_models import init_chat_model

model = init_chat_model("openai:gpt-5.4-mini")

# invoke() — single response
response = model.invoke("Why is the sky blue?")
print(response.content)

The sky appears blue primarily due to a phenomenon called Rayleigh scattering. When sunlight passes through the Earth's atmosphere, it collides with molecules and small particles in the air. Sunlight, or white light, is made up of various colors, each with different wavelengths. Blue light has a shorter wavelength compared to other colors (like red or yellow).

As sunlight interacts with the atmosphere, the shorter wavelengths (blue and violet) are scattered in all directions more effectively than the longer wavelengths (like red). Although violet light is scattered even more than blue, our eyes are more sensitive to blue light, and some of the violet light gets absorbed by the ozone layer, contributing to the blue color of the sky that we observe.

During sunset or sunrise, the light has to pass through more of the Earth's atmosphere, which scatters shorter wavelengths out of sight and allows the longer wavelengths (reds and oranges) to dominate, creating the beautiful colors typical 

In [11]:
# stream() — token by token
for chunk in model.stream("Tell me a short joke"):
    print(chunk.content, end="", flush=True)

Why don't scientists trust atoms? Because they make up everything!

In [12]:
# batch() — parallel requests
responses = model.batch(["Capital of France?", "Capital of Japan?", "Capital of Brazil?"])
for r in responses:
    print(r.content)

The capital of France is Paris.
The capital of Japan is Tokyo.
The capital of Brazil is Brasília. It was officially inaugurated as the capital on April 21, 1960, and was designed by the architect Oscar Niemeyer and urban planner Lúcio Costa.


---
## 3. Messages

LangChain uses 4 message types to represent conversations:

| Type | Role | Purpose |
|------|------|----------|
| `SystemMessage` | system | Sets model behavior/persona |
| `HumanMessage` | user | User input |
| `AIMessage` | assistant | Model response |
| `ToolMessage` | tool | Tool execution result |

You can use **message objects** or **dict format** interchangeably.

In [13]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# Using message objects
messages = [
    SystemMessage("You are a pirate. Always respond in pirate speak."),
    HumanMessage("What is machine learning?"),
]
response = model.invoke(messages)
print(response.content)

Arrr, matey! Machine learnin' be a fancy way for computin' machines to learn from data without bein’ explicitly programmed for each task, savvy? It be like a ship settin’ sail on its own course after chartin' a few navigational maps. The machines be usin' algorithms to find patterns and make decisions, just as a pirate reads the stars to find treasure. So, whether it be predictin' where the gold be buried or helpin' with tasks, machine learnin' be the compass guidin' them on their journey! Yarrr!


In [14]:
# Equivalent dict format (OpenAI-compatible)
response = model.invoke([
    {"role": "system", "content": "You are a pirate. Always respond in pirate speak."},
    {"role": "user", "content": "What is machine learning?"},
])
print(response.content)

Ahoy, matey! Machine learnin’ be like a clever parrot taught to speak, but instead o’ squawkin’ random words, it learns patterns from seas o’ data! It be a branch o’ artificial intelligence where computers be taught to make decisions or predictions without bein’ explicitly programmed for every task. 

They be siftin' through treasure troves o' information, findin’ patterns and makin' sense o' the chaos, just like us pirates lookin’ fer buried gold on a map! With the right algorithms and a heap o' data, these savvy machines can navigate the stormy seas o' complexity and help us with tasks from tradin' to navigatin' the stars. Arrr!


In [15]:
# Multi-turn conversation
conversation = [
    SystemMessage("You translate English to French."),
    HumanMessage("I love programming."),
    AIMessage("J'adore la programmation."),
    HumanMessage("I love building AI applications."),
]
response = model.invoke(conversation)
print(response.content)

J'adore créer des applications d'IA.


In [16]:
# AIMessage metadata — token usage
response = model.invoke("Hello!")
print(f"Content: {response.content}")
print(f"Type: {response.type}")
print(f"Token usage: {response.usage_metadata}")

Content: Hello! How can I assist you today?
Type: ai
Token usage: {'input_tokens': 9, 'output_tokens': 9, 'total_tokens': 18, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [17]:
# Multimodal — sending an image
multimodal_msg = HumanMessage(content=[
    {"type": "text", "text": "Describe what you see in this image in one sentence."},
    {
        "type": "image_url",
        "image_url": {"url": "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png"},
    },
])
response = model.invoke([multimodal_msg])
print(response.content)

The image features three colorful dice: a blue die, a red die, and a green die, all showing white pips.


---
## 4. Tools

Tools let agents interact with the world. Define them with the `@tool` decorator.

**Two rules:**
1. **Type hints are mandatory** — they define the schema the model sees
2. **Docstrings are mandatory** — they become the tool description

In [18]:
from langchain_core.tools import tool

@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression.

    Args:
        expression: A Python math expression like '2 + 2' or '3 ** 4'
    """
    import math
    return str(eval(expression))

# Inspect the tool
print(f"Name: {calculate.name}")
print(f"Description: {calculate.description}")
print(f"Schema: {calculate.args_schema.model_json_schema()}")

Name: calculate
Description: Evaluate a mathematical expression.

    Args:
        expression: A Python math expression like '2 + 2' or '3 ** 4'
Schema: {'description': "Evaluate a mathematical expression.\n\nArgs:\n    expression: A Python math expression like '2 + 2' or '3 ** 4'", 'properties': {'expression': {'title': 'Expression', 'type': 'string'}}, 'required': ['expression'], 'title': 'calculate', 'type': 'object'}


In [21]:
# Pydantic schema for complex inputs
from pydantic import BaseModel, Field
from typing import Literal

class WeatherQuery(BaseModel):
    """Parameters for a weather lookup."""
    location: str = Field(description="City name")
    units: Literal["celsius", "fahrenheit"] = Field(default="celsius")

@tool(args_schema=WeatherQuery)
def get_weather(location: str, units: str = "celsius") -> str:
    """Get current weather for a location."""
    # Mock implementation
    temp = 22 if units == "celsius" else 72
    return f"Weather in {location}: {temp}°{'C' if units == 'celsius' else 'F'}, Sunny"

# Agent with both tools
math_weather_agent = create_agent(
    model="openai:gpt-5.4-mini",
    tools=[calculate, get_weather],
    system_prompt="You are a helpful assistant with math and weather tools.",
)

result = math_weather_agent.invoke({"messages": "What is 15 * 37? Also, what's the weather in Lisbon?"})
result["messages"][-1].pretty_print()

================================== Ai Message ==================================

15 × 37 = 555.

Weather in Lisbon: 22°C, sunny.


### Dependency Injection with `context_schema`

Pass runtime context (database connections, user info, etc.) to tools without hardcoding.

In [30]:
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime

@dataclass
class UserContext:
    user_name: str
    role: str

USER_DB = {
    "alice": {"balance": 5000, "plan": "premium"},
    "bob": {"balance": 120, "plan": "free"},
}

@tool
def get_account_info(runtime: ToolRuntime[UserContext]) -> str:
    """Get the current user's account information."""
    name = runtime.context.user_name
    info = USER_DB.get(name, {})
    return f"User: {name} | Balance: ${info.get('balance')} | Plan: {info.get('plan')}"

account_agent = create_agent(
    model="openai:gpt-5.4-mini",
    tools=[get_account_info],
    context_schema=UserContext,
    system_prompt="You help users check their account information.",
)

# Inject different user contexts at runtime
result = account_agent.invoke(
    {"messages": "What's my name and account balance?"},
    context=UserContext(user_name="alice", role="admin"),
)


result["messages"][-1].pretty_print()

================================== Ai Message ==================================

Your name is **alice** and your account balance is **$5,000**.


---
## 5. Short-Term Memory

Add persistent conversation memory by passing a **checkpointer** to `create_agent()`.

Each conversation is identified by a unique `thread_id` — different threads = different memories.

In [31]:
from langgraph.checkpoint.memory import InMemorySaver

agent_with_memory = create_agent(
    model="openai:gpt-5.4-mini",
    tools=[search],
    system_prompt="You are a helpful assistant. Remember details from our conversation.",
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "session-1"}}

In [32]:
# First message
result = agent_with_memory.invoke(
    {"messages": "Hi! My name is Lucas and I'm a data scientist."},
    config,
)
result["messages"][-1].pretty_print()

================================== Ai Message ==================================

Hi Lucas — nice to meet you. Data science is a great field. How can I help today?


In [34]:
# Follow-up — the agent remembers!
result = agent_with_memory.invoke(
    {"messages": "What is my name and what do I do?"},
    config,
)
result["messages"][-1].pretty_print()

================================== Ai Message ==================================

Your name is Lucas, and you’re a data scientist.


In [35]:
# Different thread = different memory (no knowledge of Lucas)
config_2 = {"configurable": {"thread_id": "session-2"}}

result = agent_with_memory.invoke(
    {"messages": "What is my name?"},
    config_2,
)
result["messages"][-1].pretty_print()

================================== Ai Message ==================================

I don’t know your name yet — you haven’t told me in this conversation.

If you want, you can tell me your name and I’ll remember it for this chat.


In [36]:
# Continue the original conversation (session-1)
result = agent_with_memory.invoke(
    {"messages": "Search the web for the latest Python data science libraries in 2026."},
    config,  # back to session-1
)
result["messages"][-1].pretty_print()

================================== Ai Message ==================================

I searched the web for “latest Python data science libraries in 2026.” Here are the main libraries that showed up:

- **Polars** — fast DataFrames, often positioned as a modern alternative to pandas
- **Pandas** — still the core standard for data manipulation
- **DuckDB** — SQL analytics locally, without setting up a database
- **Scikit-learn** — classic machine learning toolkit
- **XGBoost** — strong gradient boosting library
- **Plotly** — interactive visualization
- **Great Expectations** — data quality and validation
- **MLflow** — experiment tracking and model lifecycle
- **Hugging Face Transformers** — NLP and broader transformer-based AI
- **Poetry** — dependency and environment management

A few newer or less mainstream libraries that also came up:
- **Pandera** — DataFrame validation with schemas
- **Vaex** — large dataset handling
- **ydata-profiling** — quick EDA/report generation

If you want,

---
## 6. Streaming

Three stream modes for different use cases:

| Mode | Output | Use Case |
|------|--------|----------|
| `"values"` | Full state after each step | Track agent reasoning |
| `"messages"` | `(token, metadata)` tuples | Real-time chatbot UX |
| `"custom"` | User-defined data | Progress updates from tools |

In [37]:
# Mode 1: "values" — see each agent step
for step in agent_with_memory.stream(
    {"messages": "What is the capital of Portugal?"},
    config,
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    print("---")

================================ Human Message =================================

What is the capital of Portugal?
---
================================== Ai Message ==================================

The capital of Portugal is **Lisbon**.
---


In [ ]:
# Mode 2: "messages" — token by token (ideal for chatbot UX)
for token, metadata in agent_with_memory.stream(
    {"messages": "Write a short poem about AI agents"},
    config,
    stream_mode="messages",
):
    if token.content:
        print(token.content, end="", flush=True)

In [38]:
# Mode 3: "custom" — progress updates from inside tools
from langgraph.config import get_stream_writer
import time

@tool
def research_topic(query: str) -> str:
    """Research a topic with progress updates."""
    writer = get_stream_writer()
    writer({"status": "Searching for information..."})
    time.sleep(1)
    writer({"status": "Analyzing results..."})
    time.sleep(1)
    return f"Research complete: found 5 relevant articles about '{query}'"

streaming_agent = create_agent(
    model="openai:gpt-5.4-mini",
    tools=[research_topic],
)

for chunk in streaming_agent.stream(
    {"messages": "Research the state of LLM agents in 2026"},
    stream_mode=["values", "custom"],
):
    print(chunk)

('values', {'messages': [HumanMessage(content='Research the state of LLM agents in 2026', additional_kwargs={}, response_metadata={}, id='e23d5a87-2990-4f7b-bf22-45931be652c9')]})
('values', {'messages': [HumanMessage(content='Research the state of LLM agents in 2026', additional_kwargs={}, response_metadata={}, id='e23d5a87-2990-4f7b-bf22-45931be652c9'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 136, 'total_tokens': 181, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DQAMOBayIYl4xy8FWnE6OaMpaCao6', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d4dea-c468-7722-ad7d-aa6cb935636b-0',

---
## Summary

| Concept | Key API | What it Does |
|---------|---------|-------------|
| **Agents** | `create_agent()` | Creates a LangGraph agent from model + tools |
| **Models** | `init_chat_model()` | Provider-agnostic model initialization |
| **Messages** | `SystemMessage`, `HumanMessage`, `AIMessage`, `ToolMessage` | Structured conversation format |
| **Tools** | `@tool` decorator | Define capabilities for agents |
| **Memory** | `InMemorySaver` + `thread_id` | Persistent conversation state |
| **Streaming** | `agent.stream()` with modes | Real-time output delivery |

**Next**: [Notebook 2 — Structured Outputs & Practical Applications](./2.0-structured-outputs.ipynb)